# 00 — Setup do ambiente no Google Colab

Execute no início de cada sessão do Colab (kernel remoto via extensão do VS Code, v0.2.1+).

O kernel remoto **não enxerga os arquivos locais**. Há dois modos para levar o código até ele:

- **`upload`** (desenvolvimento): no Explorer do VS Code, clique com o botão direito em `src/`, `configs/` e `pyproject.toml` → *Upload to Colab* (destino: `/content/ProjetoFinalRNAP_UnB2026/`). Reenvie após cada alteração e confira com `!ls`. Não há hash de commit rastreável.
- **`git`** (experimentos que entram no artigo): faça *commit + push* localmente; o notebook clona/atualiza o repositório público. O hash do commit é registrado nos resultados.


In [ ]:
MODE = "upload"  # "upload" ou "git"
REPO_URL = "https://github.com/Diehlgit/ProjetoFinalRNAP_UnB2026.git"  # repositório público, sem token
BRANCH = "main"
REPO_DIR = "/content/ProjetoFinalRNAP_UnB2026"

## 1. Montar o Google Drive

Também disponível pela paleta de comandos: `Colab: Mount Google Drive to Server...`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Disponibilizar o código

In [ ]:
import os, subprocess

if MODE == "git":
    if not os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                         capture_output=True, text=True).stdout)
elif MODE == "upload":
    missing = [p for p in ("src/cerrado_flora", "configs", "pyproject.toml")
               if not os.path.exists(f"{REPO_DIR}/{p}")]
    assert not missing, f"Faltam arquivos em {REPO_DIR}: {missing}. Use 'Upload to Colab'."
    print("Modo upload: resultados ficarão sem hash de commit (git_commit = None).")
else:
    raise ValueError(MODE)

## 3. Instalar o pacote (modo editável)

`torch`/`torchvision` já vêm no Colab e não são reinstalados. A célula seguinte à instalação adiciona `src/` ao `sys.path`, porque o kernel em execução não lê o `.pth` criado pela instalação editável (sem isso, `import cerrado_flora` falha com `ModuleNotFoundError` até reiniciar o ambiente). No modo `upload`, alterações em `src/` reenviadas exigem reiniciar o kernel ou `importlib.reload`.

In [ ]:
%pip install -q -e {REPO_DIR}

In [ ]:
import importlib
import sys

# A instalação editável registra o pacote num arquivo .pth, que o Python só lê ao iniciar;
# o kernel já em execução não o enxerga. Adiciona src/ ao caminho para evitar reiniciar.
SRC_DIR = f"{REPO_DIR}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
importlib.invalidate_caches()

import cerrado_flora

assert cerrado_flora.__file__.startswith(SRC_DIR), cerrado_flora.__file__
print("cerrado_flora", cerrado_flora.__version__, "em", cerrado_flora.__file__)

## 4. Verificar ambiente e caminhos

In [ ]:
import yaml
from pathlib import Path
from cerrado_flora.utils import run_metadata

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths = yaml.safe_load(f)
drive_root = Path(paths["drive_root"])
for name, sub in paths["drive_subdirs"].items():
    p = drive_root / sub
    p.mkdir(parents=True, exist_ok=True)
    print(f"{name:12s} {p}")

meta = run_metadata.collect(REPO_DIR)
print({k: v for k, v in meta.items() if k != "pip_freeze"})

## 5. Copiar o dataset para o disco local do Colab

Copia `<drive_root>/processed/cerrado_df_v1.zip` para `/content/cache/` e descompacta (ler do disco local é bem mais rápido que do Drive). Se o zip não estiver no Drive, a alternativa é recriar as imagens direto do iNaturalist com `fetch` (alguns minutos; verifica os checksums):

```
!python -m cerrado_flora.data fetch --manifests {REPO_DIR}/data/manifests --out /content/cache/cerrado_df_v1
```

No fim, confere se todas as fotos do manifesto estão presentes.

In [ ]:
import shutil
import zipfile

import pandas as pd

DATASET = "cerrado_df_v1"
cache = Path(paths["local_cache"])
data_dir = cache / DATASET
dataset_zip = drive_root / paths["drive_subdirs"]["processed"] / f"{DATASET}.zip"

if not (data_dir / "images").exists():
    assert dataset_zip.exists(), (
        f"{dataset_zip} não encontrado no Drive. Envie o zip para essa pasta "
        "ou use o comando `fetch` descrito acima.")
    cache.mkdir(parents=True, exist_ok=True)
    local_zip = cache / dataset_zip.name
    print(f"copiando {dataset_zip} ...")
    shutil.copy(dataset_zip, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(cache)  # o zip contém a pasta cerrado_df_v1/
    local_zip.unlink()

images = pd.read_csv(f"{REPO_DIR}/data/manifests/images.csv")
missing = [f for f in images["file"] if not (data_dir / f).exists()]
print(f"{len(images)} fotos no manifesto, {len(missing)} ausentes em {data_dir}")
assert not missing, missing[:5]